In [1]:
pip install transformers datasets torchaudio librosa

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ------------------ --------------------- 1.3/2.8 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 6.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/30.3 MB ? eta -:--:--
   - -------------------------------------- 1.3/30.3 MB 6.1 MB/s eta 0:00:05
   ---- ----------------------------------- 3.7/30.3 MB 8.7 MB/s eta 0:00:04
   ------- -------------------------------- 5.5/30.3 MB 9.1 MB/s eta 0:00:03
   --------- ------------------------------ 7.3/30.3 MB 9.1 MB/s eta 0:00:03
   ------------ --------------------------- 9.7/30.3 MB 9.2 MB/s eta 0:00:03
   --------------- ------------------------ 11.5/30.3 MB 9.1 MB/s eta 0:00:03
   ----------------- ---------------------- 13.6/30.3 MB 9.2 MB/s eta 0:00:02
   -------------------- ------------------- 15.7/30.3 MB 9.2 MB/s eta 0:00:02
   ----------------------- ---------------- 17.6/30.3 MB 9.1 MB/s eta 0:00:02
   ---------

In [2]:
import torch
import librosa
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

# Load processor and model
processor = Wav2Vec2Processor.from_pretrained("jonatasgrosman/wav2vec2-large-xlsr-53-english")
model = Wav2Vec2ForCTC.from_pretrained("jonatasgrosman/wav2vec2-large-xlsr-53-english")

# Load and resample the audio
audio_path = "/Users/zayna/OneDrive/Bureau/Facebook’s XLS-R/data_speechbrain.wav"
speech, sr = librosa.load(audio_path, sr=16000)

# Prepare inputs
inputs = processor(speech, sampling_rate=16000, return_tensors="pt", padding=True)

# Inference
with torch.no_grad():
    logits = model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)[0]

print("\n✅ Transcription:\n", transcription)





✅ Transcription:
 herebyaerand the kitchenis retored to hongwors epointof col te fir wather


In [3]:
import evaluate

# Load WER and CER metrics
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

# Your ground truth transcription (from image)
reference = """
mum hello I'm here okay hello hi where's the cigarettes in the kitchen the camera's on yes are you talking to it while you work no what y' doing then what's the point oh god look what I'm wearing
"""

# Model prediction (your output from earlier)
prediction = """
herebyaerand the kitchenis retored to hongwors epointof col te fir wather
"""

# Normalize and flatten text
reference = " ".join(reference.lower().split())
prediction = " ".join(prediction.lower().split())

# Compute and display metrics
wer = wer_metric.compute(predictions=[prediction], references=[reference])
cer = cer_metric.compute(predictions=[prediction], references=[reference])

print(f"WER (Word Error Rate): {wer:.2%}")
print(f"CER (Character Error Rate): {cer:.2%}")

WER (Word Error Rate): 94.87%
CER (Character Error Rate): 73.33%
